# Black-box CNN optimization with JAX and ENNx on a T4

This notebook uses the public ENNx Python API for an applied optimization loop. JAX compiles and evaluates a quantized convolutional network on the T4. ENNx keeps packed candidate history on the same GPU and performs candidate generation, exact distance, posterior scoring, acquisition, and selection with CUDA-Oxide kernels.

The optimization phase does not differentiate through the network. Its only feedback is the scalar objective returned by JAX.

## 1. Prepare a Colab T4 runtime

Select **Runtime > Change runtime type > T4 GPU** before continuing. CUDA 12 is used because it supports T4 and works with the driver line commonly assigned by Colab. See the [official JAX installation guide](https://docs.jax.dev/en/latest/installation.html) for other CUDA versions.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 12), "The ENNx Colab wheel targets CPython 3.12"
CUDA_WHEEL = (
    "https://github.com/Kvutza/ennx/releases/download/v0.1.1/"
    "ennx-0.1.1%2Bcuda75-cp312-cp312-manylinux_2_28_x86_64.whl"
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", CUDA_WHEEL],
    check=True,
)

# JAX and ENNx share the T4. Avoid reserving almost all device memory for JAX.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
import json
import platform

import jax

gpu_info = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,compute_cap,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    text=True,
).strip()
gpu_devices = [device for device in jax.devices() if device.platform == "gpu"]
runtime = {
    "python": platform.python_version(),
    "jax": jax.__version__,
    "nvidia_smi": gpu_info,
    "jax_devices": [str(device) for device in jax.devices()],
}
print(json.dumps(runtime, indent=2))
assert gpu_devices, "JAX did not discover a GPU; reconnect with a T4 runtime"
GPU = gpu_devices[0]

## 2. Import the CUDA-enabled ENNx extension

The released wheel contains the CPython 3.12 extension and CUDA-Oxide kernels compiled for T4 `sm_75`. No Rust, LLVM, or CUDA compiler toolchain is needed in this tutorial.

In [ ]:
from ennx.experimental import PackedSearch

print(PackedSearch)

## 3. Understand the dataflow

Each CNN parameter is represented by an affine unsigned int4 code. Subtracting the fixed zero point during JAX decoding makes the represented values signed; pairwise distances are unchanged by that common offset.

For each optimization round:

1. Python supplies deterministic candidate seeds to `PackedSearch.ask`.
2. ENNx materializes and scores packed candidates on the T4 and returns one selected trial.
3. `PackedSearch.row()` transfers only that selected packed row to Python.
4. JAX decodes the row, evaluates the CNN on the T4, and returns a scalar objective.
5. `PackedSearch.tell` records the observation and optionally moves the incumbent center.

Step 3 is the remaining host boundary. Candidate-by-history matrices and rejected candidate rows never leave the GPU.

## 4. Create a self-contained image task

The objective classifies noisy vertical, horizontal, and diagonal bars. It is small enough for a tutorial but still exercises convolution, nonlinear activation, and a dense classifier.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax import lax, random

IMAGE_SIZE = 12
NUM_CLASSES = 3


def make_dataset(key, examples):
    label_key, offset_key, noise_key = random.split(key, 3)
    labels = random.randint(label_key, (examples,), 0, NUM_CLASSES)
    offsets = random.randint(offset_key, (examples,), -2, 3).astype(jnp.float32)
    axis = jnp.arange(IMAGE_SIZE, dtype=jnp.float32)
    yy, xx = jnp.meshgrid(axis, axis, indexing="ij")
    center = (IMAGE_SIZE - 1) / 2 + offsets[:, None, None]
    vertical = jnp.abs(xx[None, :, :] - center) <= 1.25
    horizontal = jnp.abs(yy[None, :, :] - center) <= 1.25
    diagonal = jnp.abs((xx - yy)[None, :, :] - offsets[:, None, None]) <= 1.25
    patterns = jnp.where(
        labels[:, None, None] == 0,
        vertical,
        jnp.where(labels[:, None, None] == 1, horizontal, diagonal),
    )
    noise = 0.35 * random.normal(noise_key, patterns.shape)
    images = jnp.clip(patterns.astype(jnp.float32) + noise, 0.0, 1.0)
    return jax.device_put(images[..., None], GPU), jax.device_put(labels, GPU)


images, labels = make_dataset(random.PRNGKey(4), 512)
print(images.shape, labels.shape, images.devices())

fig, axes = plt.subplots(1, 6, figsize=(9, 2))
for axis, image, label in zip(axes, np.asarray(images[:6]), np.asarray(labels[:6])):
    axis.imshow(image[..., 0], cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"class {label}")
    axis.axis("off")
plt.tight_layout()

## 5. Define the JAX CNN and black-box objective

The scalar reward is negative cross-entropy, so larger is better. Accuracy is tracked only for interpretation. `jax.jit` compiles the complete evaluation onto the T4.

In [ ]:
FEATURES = 4
FLAT_FEATURES = 6 * 6 * FEATURES
PARAMETER_SHAPES = (
    (3, 3, 1, FEATURES),
    (FEATURES,),
    (FLAT_FEATURES, NUM_CLASSES),
    (NUM_CLASSES,),
)


def initialize_params(key):
    conv_key, dense_key = random.split(key)
    return (
        random.normal(conv_key, PARAMETER_SHAPES[0]) * jnp.sqrt(2.0 / 9.0),
        jnp.zeros(PARAMETER_SHAPES[1], dtype=jnp.float32),
        random.normal(dense_key, PARAMETER_SHAPES[2]) * jnp.sqrt(2.0 / FLAT_FEATURES),
        jnp.zeros(PARAMETER_SHAPES[3], dtype=jnp.float32),
    )


@jax.jit
def cnn(params, batch):
    conv_weight, conv_bias, dense_weight, dense_bias = params
    hidden = lax.conv_general_dilated(
        batch,
        conv_weight,
        window_strides=(2, 2),
        padding="SAME",
        dimension_numbers=("NHWC", "HWIO", "NHWC"),
    )
    hidden = jax.nn.relu(hidden + conv_bias)
    return hidden.reshape((hidden.shape[0], -1)) @ dense_weight + dense_bias


@jax.jit
def objective(params, batch, targets):
    logits = cnn(params, batch)
    log_probs = jax.nn.log_softmax(logits)
    reward = jnp.mean(log_probs[jnp.arange(targets.size), targets])
    accuracy = jnp.mean(jnp.argmax(logits, axis=1) == targets)
    return reward, accuracy


def evaluate(params):
    reward, accuracy = objective(params, images, labels)
    reward.block_until_ready()
    return float(reward), float(accuracy)


initial_params = tuple(
    jax.device_put(value, GPU) for value in initialize_params(random.PRNGKey(8))
)
initial_reward, initial_accuracy = evaluate(initial_params)
print(
    f"random float32 model: reward={initial_reward:.4f} accuracy={initial_accuracy:.3f}"
)

## 6. Pack the CNN into ENNx leaves

Each tensor gets its own quantization scale and distance weight. A leaf radius equal to one quantization step means that `length=0.08` mutates roughly eight percent of its codes by one step. The dense tensor is normalized by its element count so it does not overwhelm smaller tensors in the distance metric.

In [ ]:
ZERO_POINT = 8


def pack_codes(codes):
    codes = np.asarray(codes, dtype=np.uint8).reshape(-1)
    if codes.size % 2:
        codes = np.pad(codes, (0, 1))
    return codes[0::2] | (codes[1::2] << np.uint8(4))


def encode_params(params):
    rows = []
    leaves = []
    specs = []
    element_offset = 0
    for value in params:
        host = np.asarray(value, dtype=np.float32)
        length = host.size
        scale = max(float(np.max(np.abs(host))) / 7.0, 1.0e-3)
        codes = np.clip(np.rint(host.reshape(-1) / scale) + ZERO_POINT, 0, 15)
        rows.append(pack_codes(codes))
        leaves.append((element_offset, length, 4, scale, 1.0 / length, scale))
        specs.append((host.shape, length, scale))
        element_offset += length
    return np.concatenate(rows).astype(np.uint8), leaves, specs


def decode_params(row, specs):
    row = np.asarray(row, dtype=np.uint8)
    params = []
    byte_offset = 0
    for shape, length, scale in specs:
        byte_length = (length + 1) // 2
        packed = row[byte_offset : byte_offset + byte_length]
        codes = np.empty(byte_length * 2, dtype=np.uint8)
        codes[0::2] = packed & np.uint8(0x0F)
        codes[1::2] = packed >> np.uint8(4)
        values = (codes[:length].astype(np.float32) - ZERO_POINT) * scale
        params.append(jax.device_put(values.reshape(shape), GPU))
        byte_offset += byte_length
    assert byte_offset == row.size
    return tuple(params)


base_row, leaves, specs = encode_params(initial_params)
quantized_params = decode_params(base_row, specs)
base_reward, base_accuracy = evaluate(quantized_params)
parameter_count = sum(spec[1] for spec in specs)
print(
    f"parameters={parameter_count} packed_bytes={base_row.nbytes} "
    f"reward={base_reward:.4f} accuracy={base_accuracy:.3f}"
)

## 7. Run direct black-box optimization

`PackedSearch` is the applied optimizer here, not an inner gradient-training helper. It receives only packed parameters and scalar rewards. Every rejected evaluation remains useful posterior history, while `accept=True` moves the center used for future perturbations.

In [ ]:
import time

HISTORY_CAPACITY = 32
CANDIDATES = 512
ROUNDS = 32
LENGTH = 0.08

search = PackedSearch(
    base_row,
    base_reward,
    leaves,
    HISTORY_CAPACITY,
    device="cuda",
)
assert search.row_bytes == base_row.size

seed_rng = np.random.default_rng(19)
best_reward = base_reward
best_accuracy = base_accuracy
best_row = base_row.copy()
records = []

for round_index in range(ROUNDS):
    seeds = seed_rng.integers(
        0, np.iinfo(np.uint64).max, size=CANDIDATES, dtype=np.uint64
    )
    neighbors = min(8, search.history_len)

    proposal_start = time.perf_counter()
    candidate_index, candidate_seed, predicted_score = search.ask(
        seeds,
        LENGTH,
        neighbors,
        acquisition="thompson",
        seed=round_index,
    )
    candidate_row = np.asarray(search.row(), dtype=np.uint8)
    proposal_seconds = time.perf_counter() - proposal_start

    evaluation_start = time.perf_counter()
    reward, accuracy = evaluate(decode_params(candidate_row, specs))
    evaluation_seconds = time.perf_counter() - evaluation_start

    accepted = reward > best_reward
    search.tell(reward, accepted)
    if accepted:
        best_reward = reward
        best_accuracy = accuracy
        best_row = candidate_row.copy()

    records.append(
        {
            "round": round_index,
            "candidate": candidate_index,
            "seed": candidate_seed,
            "predicted": predicted_score,
            "reward": reward,
            "accuracy": accuracy,
            "best_reward": best_reward,
            "accepted": accepted,
            "proposal_ms": proposal_seconds * 1_000,
            "evaluation_ms": evaluation_seconds * 1_000,
        }
    )
    if accepted or round_index % 4 == 0:
        print(
            f"round={round_index:02d} reward={reward:.4f} accuracy={accuracy:.3f} "
            f"accepted={accepted} best={best_reward:.4f}"
        )

print(
    f"baseline reward={base_reward:.4f}, final reward={best_reward:.4f}; "
    f"baseline accuracy={base_accuracy:.3f}, final accuracy={best_accuracy:.3f}"
)

In [ ]:
rounds = np.asarray([record["round"] for record in records])
rewards = np.asarray([record["reward"] for record in records])
best_rewards = np.asarray([record["best_reward"] for record in records])
proposal_ms = np.asarray([record["proposal_ms"] for record in records])
evaluation_ms = np.asarray([record["evaluation_ms"] for record in records])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rounds, rewards, "o", alpha=0.55, label="observed")
axes[0].plot(rounds, best_rewards, linewidth=2, label="incumbent")
axes[0].axhline(base_reward, color="black", linestyle="--", label="baseline")
axes[0].set(xlabel="round", ylabel="negative cross-entropy")
axes[0].legend()

axes[1].plot(rounds, proposal_ms, label="ENN proposal + selected-row transfer")
axes[1].plot(rounds, evaluation_ms, label="JAX evaluation")
axes[1].set(xlabel="round", ylabel="milliseconds")
axes[1].legend()
fig.tight_layout()

## 8. Interpret the result

This is already a direct neural-network optimizer: candidate selection depends on the task reward, not on gradients or a differentiable training loss. The CUDA-resident ENNx path avoids materializing the full candidate-by-history distance matrix and transfers only the selected packed row.

The next systems step is a device-to-device handoff from the ENNx resident row buffer into a JAX custom call. That would remove `PackedSearch.row()` and make proposal, decoding, and objective evaluation one GPU-resident pipeline. The timing panel above isolates the cost that such an integration should eliminate.